# Class 26: The ReAct Loop — Reasoning + Acting, এবং সেটাকে নির্ভরযোগ্য বানানো

এই notebook সম্পূর্ণ **self-contained** — project folder লাগে না। Ollama লাগবে: `ollama serve`, `ollama pull llama3.1:8b`।

আমরা ধাপে ধাপে একটা travel agent-এর ReAct loop বানাব, প্রতিটা ধাপে live দেখব llama3.1:8b কোথায় ভাঙে, আর code দিয়ে সেটা সারাব।

| Section | কী শিখব |
| --- | --- |
| 1-2 | Classic text ReAct: prompt, stop sequence, parser, loop |
| 3-4 | Native tool calling: naive loop কেন chain করতে পারে না, template রহস্য, `final_answer` fix |
| 5 | পাঁচটা typed exit — scripted fake LLM দিয়ে |
| 6-7 | Validation + approval gate |
| 8 | Determinism experiment: temperature 0 বনাম 0.8 |
| 9 | Mini evaluation: end-state grading |
| 10 | ReWOO-style plan-first বনাম ReAct |

In [1]:
import json, re, time
from collections import Counter
from typing import Literal

import ollama
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

MODEL = "llama3.1:8b"
OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 400}
client = ollama.Client(host="http://localhost:11434")

FLIGHTS = [
    {"id": "BG101", "origin": "DAC", "destination": "CXB", "date": "2026-11-05", "departs": "08:00", "price_bdt": 6500, "seats_left": 5},
    {"id": "BG102", "origin": "DAC", "destination": "CXB", "date": "2026-11-05", "departs": "16:30", "price_bdt": 7200, "seats_left": 0},
    {"id": "US103", "origin": "DAC", "destination": "CXB", "date": "2026-11-05", "departs": "12:10", "price_bdt": 6500, "seats_left": 2},
    {"id": "BG201", "origin": "DAC", "destination": "CGP", "date": "2026-11-05", "departs": "09:15", "price_bdt": 5400, "seats_left": 12},
    {"id": "BG501", "origin": "CXB", "destination": "DAC", "date": "2026-11-08", "departs": "10:00", "price_bdt": 6600, "seats_left": 4},
]
AIRPORTS = sorted({f["origin"] for f in FLIGHTS} | {f["destination"] for f in FLIGHTS})
TASK = "Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim"
print(len(FLIGHTS), "flights; airports:", AIRPORTS)

5 flights; airports: ['CGP', 'CXB', 'DAC']


### Deterministic tools

Pure function, total-order sort `(price, departs, id)` — BG101 আর US103 দুটোরই দাম 6500, তবু ক্রম কখনো বদলায় না। Booking ref input থেকে derive হয়, `uuid4()` না। State একটা dict-এ, যাতে শেষে assert করা যায়।

In [2]:
STATE = {"bookings": []}

def search_flights(origin, destination, date=None):
    if origin not in AIRPORTS or destination not in AIRPORTS:
        return {"ok": False, "error": "unknown_airport", "known_airports": AIRPORTS}
    rows = [f for f in FLIGHTS if f["origin"] == origin and f["destination"] == destination
            and (date is None or f["date"] == date)]
    rows.sort(key=lambda f: (f["price_bdt"], f["departs"], f["id"]))
    return {"ok": True, "flights": [{k: f[k] for k in ("id", "departs", "price_bdt", "seats_left")} for f in rows]}

def book_flight(flight_id, passenger_name, approve=lambda d: True):
    f = next((x for x in FLIGHTS if x["id"] == flight_id), None)
    if f is None:
        return {"ok": False, "error": "unknown_flight", "hint": "Use an id from search_flights."}
    if f["seats_left"] - sum(b["flight_id"] == flight_id for b in STATE["bookings"]) <= 0:
        return {"ok": False, "error": "sold_out", "hint": "Pick another flight."}
    ref = f"BK-{flight_id}-{passenger_name.upper().replace(' ', '-')}"
    if any(b["ref"] == ref for b in STATE["bookings"]):
        return {"ok": False, "error": "duplicate_booking", "ref": ref}
    if not approve({"flight_id": flight_id, "passenger": passenger_name, "price_bdt": f["price_bdt"]}):
        return {"ok": False, "error": "approval_denied", "hint": "Nothing booked; tell the user."}
    STATE["bookings"].append({"ref": ref, "flight_id": flight_id, "passenger": passenger_name})
    return {"ok": True, "booking_ref": ref, "flight_id": flight_id, "passenger": passenger_name}

print(search_flights("DAC", "CXB", "2026-11-05"))
print(search_flights("DAC", "CXB", "2026-11-05") == search_flights("DAC", "CXB", "2026-11-05"), "<- same input, same output")

{'ok': True, 'flights': [{'id': 'BG101', 'departs': '08:00', 'price_bdt': 6500, 'seats_left': 5}, {'id': 'US103', 'departs': '12:10', 'price_bdt': 6500, 'seats_left': 2}, {'id': 'BG102', 'departs': '16:30', 'price_bdt': 7200, 'seats_left': 0}]}
True <- same input, same output


## 1. Classic Text ReAct — আর stop sequence কেন লাগে

২০২২-এর format: model লেখে `Thought:` / `Action:` / `Action Input:`। আমরা action চালিয়ে `Observation:` বসাই। বিপদ: generation না থামালে model নিজেই `Observation: ...` লিখে ফেলতে পারে — মানে tool result *বানিয়ে* নেয়। Ollama-র `options={"stop": ["\nObservation:"]}` সেটা আটকায়। নিচে একই prompt দুইভাবে চালাই আর দেখি এই model আসলে কী করে।

In [3]:
TEXT_SYSTEM = '''You are a travel agent. Tools:
- search_flights(origin, destination, date): airport codes like DAC; date YYYY-MM-DD. Returns flights cheapest first.
- book_flight(flight_id, passenger_name): book one seat. Only use an id from search results.
Use EXACTLY this format, one action per reply:
Thought: <reasoning>
Action: <tool name>
Action Input: <JSON object>
You will then get "Observation: <result>". When done:
Thought: <reasoning>
Final Answer: <answer>'''

msgs = [{"role": "system", "content": TEXT_SYSTEM}, {"role": "user", "content": TASK}]
no_stop = client.chat(model=MODEL, messages=msgs, options=OPTIONS).message.content
with_stop = client.chat(model=MODEL, messages=msgs, options={**OPTIONS, "stop": ["\nObservation:", "Observation:"]}).message.content
print("=== WITHOUT stop ===\n", no_stop[:700])
print("\n=== WITH stop ===\n", with_stop)
print("\nmodel invented its own Observation without stop?", "Observation" in no_stop)

=== WITHOUT stop ===
 Thought: First, I need to find the cheapest flights from DAC to CXB on 2026-11-05.
Action: search_flights
Action Input: {"origin": "DAC", "destination": "CXB", "date": "2026-11-05"}

=== WITH stop ===
 Thought: First, I need to find the cheapest flights from DAC to CXB on 2026-11-05.
Action: search_flights
Action Input: {"origin": "DAC", "destination": "CXB", "date": "2026-11-05"}

model invented its own Observation without stop? False


In [4]:
_ACTION = re.compile(r"Action:\s*`?([\w-]+)`?\s*\n+\s*Action Input:\s*(.*)", re.S)
_FINAL = re.compile(r"Final Answer:\s*(.*)", re.S)
_THOUGHT = re.compile(r"Thought:\s*(.*?)(?=\n\s*(?:Action|Final Answer):|\Z)", re.S)

def parse_text_step(text):
    thought = (m.group(1).strip() if (m := _THOUGHT.search(text)) else "")
    if a := _ACTION.search(text):
        raw = a.group(2).strip().removeprefix("```json").removeprefix("```")
        j = re.search(r"\{.*\}", raw, re.S)
        return {"thought": thought, "action": a.group(1), "input": json.loads(j.group(0)) if j else {}}
    if f := _FINAL.search(text):
        return {"thought": thought, "final": f.group(1).strip()}
    raise ValueError("no Action/Final Answer found")

print(parse_text_step(with_stop))

{'thought': 'First, I need to find the cheapest flights from DAC to CXB on 2026-11-05.', 'action': 'search_flights', 'input': {'origin': 'DAC', 'destination': 'CXB', 'date': '2026-11-05'}}


**আমাদের run-এ কী হলো:** stop ছাড়াও llama3.1 নিজে থেকে `Action Input`-এর পরে থেমেছে — Ollama-র chat template turn শেষে `<|eot_id|>` বসায়, তাই এই model-এ নকল Observation দেখা যায়নি। তবু stop sequence রাখুন: raw completion API, অন্য template বা অন্য model-এ এই guard-ই নকল observation আটকায় — আর এর খরচ শূন্য।

## 2. একটা Minimal Text ReAct Loop

Observation যায় একটা `user` message হিসেবে (`"Observation: {...}"`)। Parse ব্যর্থ হলে crash না — parse error-টাই observation।

In [5]:
TOOLS_FN = {"search_flights": search_flights, "book_flight": book_flight}

def call_tool(name, args):
    # Never raises: unknown tool or bad arguments become an observation the model can read.
    if name not in TOOLS_FN:
        return {"ok": False, "error": "unknown_tool", "valid": list(TOOLS_FN)}
    try:
        return TOOLS_FN[name](**{k: v for k, v in (args or {}).items() if v is not None})
    except TypeError as e:
        return {"ok": False, "error": "bad_arguments", "msg": str(e)}

def text_react(task, max_steps=6, verbose=True):
    STATE["bookings"].clear()
    msgs = [{"role": "system", "content": TEXT_SYSTEM}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        out = client.chat(model=MODEL, messages=msgs, options={**OPTIONS, "stop": ["\nObservation:", "Observation:"]}).message.content
        msgs.append({"role": "assistant", "content": out})
        try:
            p = parse_text_step(out)
        except (ValueError, json.JSONDecodeError) as e:
            obs = {"ok": False, "error": "parse_error", "msg": str(e)}
        else:
            if "final" in p:
                return p["final"], step
            obs = call_tool(p["action"], p["input"])
            if verbose:
                print(f"  [{step}] Thought: {p['thought'][:90]}\n      Action: {p['action']} {p['input']}\n      Observation: {json.dumps(obs)[:110]}")
        msgs.append({"role": "user", "content": "Observation: " + json.dumps(obs)})
    return "Stopped: max_steps", max_steps

answer, steps = text_react(TASK)
print("\nFINAL:", answer, f"({steps} LLM calls)  bookings:", STATE["bookings"])

  [1] Thought: First, I need to find the cheapest flights from DAC to CXB on 2026-11-05.
      Action: search_flights {'origin': 'DAC', 'destination': 'CXB', 'date': '2026-11-05'}
      Observation: {"ok": true, "flights": [{"id": "BG101", "departs": "08:00", "price_bdt": 6500, "seats_left": 5}, {"id": "US10


  [2] Thought: The cheapest flights are BG101 and US103, both priced at 6500 BDT. I'll book one seat on t
      Action: book_flight {'flight_id': 'BG101', 'passenger_name': 'Rahim'}
      Observation: {"ok": true, "booking_ref": "BK-BG101-RAHIM", "flight_id": "BG101", "passenger": "Rahim"}



FINAL: The booking reference for Rahim's flight from DAC to CXB on 2026-11-05 is BK-BG101-RAHIM. (3 LLM calls)  bookings: [{'ref': 'BK-BG101-RAHIM', 'flight_id': 'BG101', 'passenger': 'Rahim'}]


## 3. Native Tool Calling — naive loop

এবার action যায় Ollama-র structured `tool_calls` channel-এ। Model আর observation বানাতে পারে না, কারণ result ঢোকে শুধু `role="tool"` message দিয়ে। Class 25-এর মতো সাধারণ loop লিখি — দেখি কী হয়।

In [6]:
SCHEMAS = [
    {"type": "function", "function": {"name": "search_flights",
        "description": "Search flights, cheapest first. Always search before booking.",
        "parameters": {"type": "object", "properties": {
            "origin": {"type": "string", "description": "IATA code e.g. DAC"},
            "destination": {"type": "string", "description": "IATA code e.g. CXB"},
            "date": {"type": "string", "description": "YYYY-MM-DD, only if the user gave one"}},
            "required": ["origin", "destination"]}}},
    {"type": "function", "function": {"name": "book_flight",
        "description": "Book ONE seat. Only use a flight id from search results.",
        "parameters": {"type": "object", "properties": {
            "flight_id": {"type": "string"}, "passenger_name": {"type": "string"}},
            "required": ["flight_id", "passenger_name"]}}},
]
NATIVE_SYSTEM = ("You are a travel agent. Use the tools. Always search before booking. Only book when the user asked to book. "
                 "When the user asked to book and gave a name, call book_flight directly — the system asks the user for approval.")

def naive_native(task, max_steps=6):
    STATE["bookings"].clear()
    msgs = [{"role": "system", "content": NATIVE_SYSTEM}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        m = client.chat(model=MODEL, messages=msgs, tools=SCHEMAS, options=OPTIONS).message
        if not m.tool_calls:
            return m.content, step
        msgs.append(m)
        for c in m.tool_calls:
            obs = call_tool(c.function.name, c.function.arguments)
            print(f"  [{step}] {c.function.name}({c.function.arguments}) -> {json.dumps(obs)[:90]}")
            msgs.append({"role": "tool", "tool_name": c.function.name, "content": json.dumps(obs)})
    return "Stopped: max_steps", max_steps

answer, steps = naive_native(TASK)
print("\nFINAL:", answer, f"({steps} LLM calls)  bookings:", STATE["bookings"])

  [1] search_flights({'origin': 'DAC', 'date': '2026-11-05', 'destination': 'CXB'}) -> {"ok": true, "flights": [{"id": "BG101", "departs": "08:00", "price_bdt": 6500, "seats_lef



FINAL: The cheapest DAC to CXB flight on 2026-11-05 is BG101, departing at 08:00, with a price of 6500 BDT. There are 5 seats left. Would you like to book this flight for Rahim? (2 LLM calls)  bookings: []


Search হলো, কিন্তু **book হলো না** — model "book করব" বা "confirm করবেন?" লিখে থেমে গেছে। কেন? llama3.1-এর chat template দেখি:

In [7]:
tpl = client.show(MODEL).template
i = tpl.find('if eq .Role "user"')
print(tpl[i:i + 520])

if eq .Role "user" }}<|start_header_id|>user<|end_header_id|>
{{- if and $.Tools $last }}

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{{ range $.Tools }}
{{- . }}
{{ end }}
Question: {{ .Content }}<|eot_id|>
{{- else }}

{{ .Content }}<|eot_id|>
{{- end }}{{ if $last }}<|start_header_id|>assist


দুটো জিনিস:

1. `{{- if and $.Tools $last }}` — tool list দেখানো হয় **শুধু শেষ message যদি user হয়**। Tool result আসার পর শেষ message `tool` — model তার tools দেখতেই পায় না, তাই আর `book_flight` ডাকতে পারে না।
2. Tool দেখানো হলে লেখা থাকে *"please respond with a JSON for a function call"* — তখন model plain text-এ থামতে পারে না।

## 4. Fix: `final_answer` tool + nudge

smolagents-এর `ToolCallingAgent`-এর একই idea: থামাটাও একটা tool call। আর প্রতিটা observation-এর পরে একটা user nudge — যাতে tools আবার দেখা যায় — যেটা **মূল request** আর **ইতিমধ্যে করা কাজের তালিকা** দেয় (template nudge-কে `Question:` বানায়, request না দিলে model nudge-টাকেই প্রশ্ন ভাবে)।

In [8]:
FINAL_TOOL = {"type": "function", "function": {"name": "final_answer",
    "description": "Give the final reply to the user and end. Use when done, when you must ask the user something, or when it cannot be done.",
    "parameters": {"type": "object", "properties": {"answer": {"type": "string"}}, "required": ["answer"]}}}

def nudge(task, done_lines):
    return (f"My request: {task}\nAlready done (do not repeat these):\n" + ("\n".join(done_lines) or "(nothing yet)") +
            "\nIf my request still needs an action that is not done yet, call that tool now. Otherwise call final_answer with your reply to me.")

def fixed_native(task, max_steps=6):
    STATE["bookings"].clear()
    msgs = [{"role": "system", "content": NATIVE_SYSTEM + " When finished, call final_answer."}, {"role": "user", "content": task}]
    done = []
    for step in range(1, max_steps + 1):
        m = client.chat(model=MODEL, messages=msgs, tools=SCHEMAS + [FINAL_TOOL], options=OPTIONS).message
        calls = m.tool_calls or []
        if not calls:
            return m.content, step
        if fin := next((c for c in calls if c.function.name == "final_answer"), None):
            return fin.function.arguments.get("answer", ""), step
        msgs.append(m)
        for c in calls:
            obs = call_tool(c.function.name, c.function.arguments)
            print(f"  [{step}] {c.function.name}({c.function.arguments}) -> {json.dumps(obs)[:90]}")
            msgs.append({"role": "tool", "tool_name": c.function.name, "content": json.dumps(obs)})
            done.append(f"{len(done) + 1}. {c.function.name}({c.function.arguments}) -> {json.dumps(obs)[:160]}")
        msgs.append({"role": "user", "content": nudge(task, done)})
    return "Stopped: max_steps", max_steps

answer, steps = fixed_native(TASK)
print("\nFINAL:", answer, f"({steps} LLM calls)  bookings:", STATE["bookings"])

  [1] search_flights({'date': '2026-11-05', 'destination': 'CXB', 'origin': 'DAC'}) -> {"ok": true, "flights": [{"id": "BG101", "departs": "08:00", "price_bdt": 6500, "seats_lef


  [2] book_flight({'flight_id': 'BG101', 'passenger_name': 'Rahim'}) -> {"ok": true, "booking_ref": "BK-BG101-RAHIM", "flight_id": "BG101", "passenger": "Rahim"}


  [3] book_flight({'flight_id': 'BG101', 'passenger_name': 'Rahim'}) -> {"ok": false, "error": "duplicate_booking", "ref": "BK-BG101-RAHIM"}



FINAL: Flight booked successfully for Rahim on BG101. (4 LLM calls)  bookings: [{'ref': 'BK-BG101-RAHIM', 'flight_id': 'BG101', 'passenger': 'Rahim'}]


## 5. পাঁচটা Typed Exit — scripted fake LLM দিয়ে

ReAct paper-এর error analysis: failure-এর **৪৭% reasoning loop**, ২৩% অকেজো search, hallucination 0%। তাই loop-এ guard লাগে, আর প্রতিটা guard একটা আলাদা `status` দেয় — test বলতে পারে *কেন* থামল। LLM-কে একটা callable হিসেবে inject করি, যাতে fake দিয়ে প্রতিটা exit হুবহু ঘটানো যায়।

In [9]:
MISTAKES = {"unknown_tool", "invalid_arguments", "repeated_call"}

def guarded_loop(task, llm, execute, max_steps=8, max_repeats=2, max_errors=3, deadline=180, clock=time.monotonic):
    msgs = [{"role": "user", "content": task}]
    seen, errors, start, steps = {}, 0, clock(), []
    for n in range(1, max_steps + 1):
        if clock() - start > deadline:
            return "timeout", steps
        reply = llm(msgs)
        calls = reply.get("tool_calls") or []
        if not calls:
            return "final", steps
        for name, args in calls:
            key = name + json.dumps(args, sort_keys=True)         # key-এর ক্রম model বদলায়, তাই sort
            seen[key] = seen.get(key, 0) + 1
            if seen[key] > max_repeats:
                return "loop", steps
            obs = execute(name, args) if seen[key] == 1 else {"ok": False, "error": "repeated_call",
                                                               "hint": "You already did this; use the earlier result."}
            steps.append((name, args, obs.get("error", "ok")))
            errors = errors + 1 if obs.get("error") in MISTAKES else 0
            msgs.append({"role": "tool", "content": json.dumps(obs)})
        if errors >= max_errors:
            return "error_budget", steps
    return "max_steps", steps

def scripted(*replies):
    it = iter(replies)
    return lambda msgs: next(it)

ok_exec = lambda name, args: ({"ok": True} if name in TOOLS_FN else {"ok": False, "error": "unknown_tool"})
S = ("search_flights", {"origin": "DAC", "destination": "CXB"})
S_swapped = ("search_flights", {"destination": "CXB", "origin": "DAC"})
ticks = iter([0, 0, 0, 999])
cases = {
    "answers directly": guarded_loop("x", scripted({"content": "hi"}), ok_exec),
    "same call x3 (keys reordered)": guarded_loop("x", scripted(*[{"tool_calls": [c]} for c in (S, S_swapped, S)]), ok_exec),
    "unknown tool x3": guarded_loop("x", scripted(*[{"tool_calls": [("fly", {"n": i})]} for i in range(3)]), ok_exec),
    "never stops": guarded_loop("x", scripted(*[{"tool_calls": [("search_flights", {"p": i})]} for i in range(9)]), ok_exec),
    "slow world": guarded_loop("x", scripted(*[{"tool_calls": [("search_flights", {"p": i})]} for i in range(9)]), ok_exec,
                               clock=lambda: next(ticks)),
}
for name, (status, steps) in cases.items():
    print(f"{name:<32} -> {status:<13} after {len(steps)} action(s)")

answers directly                 -> final         after 0 action(s)
same call x3 (keys reordered)    -> loop          after 2 action(s)
unknown tool x3                  -> error_budget  after 3 action(s)
never stops                      -> max_steps     after 8 action(s)
slow world                       -> timeout       after 2 action(s)


## 6. Validation — error-ই নতুন তথ্য

Temperature 0-এ একই prompt আবার পাঠালে model সম্ভবত একই ভুল করবে। যা বদলায় তা হলো **error observation** — field-এর নাম আর hint। Pydantic: lax mode `"6600"`-কে int বানায়, validator `"dac"`-কে `"DAC"` করে, `extra="forbid"` বানানো field ধরে।

In [10]:
class SearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    origin: str
    destination: str
    date: str | None = None

    @field_validator("origin", "destination")
    @classmethod
    def iata(cls, v):
        v = v.strip().upper()
        if not re.fullmatch(r"[A-Z]{3}", v):
            raise ValueError("must be a 3-letter airport code like DAC")
        return v

    @field_validator("date")
    @classmethod
    def iso(cls, v):
        if v and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", v):
            raise ValueError("must be YYYY-MM-DD")
        return v

def execute_search(raw):
    try:
        a = SearchArgs.model_validate(raw)
    except ValidationError as e:
        return {"ok": False, "error": "invalid_arguments",
                "details": [{"field": ".".join(map(str, d["loc"])), "msg": d["msg"]} for d in e.errors()],
                "hint": "Correct the arguments and call again. Do not invent values; ask the user if unsure."}
    return search_flights(**a.model_dump())

for raw in [{"from": "Dhaka", "to": "Cox's Bazar"}, {"origin": "dac", "destination": "cxb", "date": "5 Nov"},
            {"origin": "dac", "destination": "cxb", "date": "2026-11-05"}]:
    print(raw, "\n   ->", json.dumps(execute_search(raw))[:200], "\n")

{'from': 'Dhaka', 'to': "Cox's Bazar"} 
   -> {"ok": false, "error": "invalid_arguments", "details": [{"field": "origin", "msg": "Field required"}, {"field": "destination", "msg": "Field required"}, {"field": "from", "msg": "Extra inputs are not  

{'origin': 'dac', 'destination': 'cxb', 'date': '5 Nov'} 
   -> {"ok": false, "error": "invalid_arguments", "details": [{"field": "date", "msg": "Value error, must be YYYY-MM-DD"}], "hint": "Correct the arguments and call again. Do not invent values; ask the user  

{'origin': 'dac', 'destination': 'cxb', 'date': '2026-11-05'} 
   -> {"ok": true, "flights": [{"id": "BG101", "departs": "08:00", "price_bdt": 6500, "seats_left": 5}, {"id": "US103", "departs": "12:10", "price_bdt": 6500, "seats_left": 2}, {"id": "BG102", "departs": "1 



## 7. Approval Gate — code-এ, prompt-এ না

`book_flight` irreversible। Prompt-এ "আগে confirm করো" বলা যায়, কিন্তু enforce করে শুধু code: `approve(details)` callback — CLI-তে `input()`, UI-তে বাটন, এখানে একটা "human" যে শুধু নিজের বলা নামের booking মানে।

In [11]:
STATE["bookings"].clear()
human = lambda d: d["passenger"] in ("Rahim",)
print(book_flight("BG101", "John Doe", approve=human))     # model নাম বানালে
print(book_flight("BG101", "Rahim", approve=human))
print(book_flight("BG101", "Rahim", approve=human))        # idempotent
print(book_flight("BG102", "Rahim", approve=human))        # tool নিজেই precondition check করে
print("bookings:", STATE["bookings"])

{'ok': False, 'error': 'approval_denied', 'hint': 'Nothing booked; tell the user.'}
{'ok': True, 'booking_ref': 'BK-BG101-RAHIM', 'flight_id': 'BG101', 'passenger': 'Rahim'}
{'ok': False, 'error': 'duplicate_booking', 'ref': 'BK-BG101-RAHIM'}
{'ok': False, 'error': 'sold_out', 'hint': 'Pick another flight.'}
bookings: [{'ref': 'BK-BG101-RAHIM', 'flight_id': 'BG101', 'passenger': 'Rahim'}]


## 8. Determinism Experiment

একই task ৩ বার, temperature 0 আর temperature 0.8 (Ollama-র default!)-এ। প্রতিটা run-এর trajectory (tool call-এর ক্রম) তুলনা করি।

In [12]:
def trajectory(temperature, seed):
    STATE["bookings"].clear()
    opts = {**OPTIONS, "temperature": temperature, "seed": seed}
    msgs = [{"role": "system", "content": NATIVE_SYSTEM + " When finished, call final_answer."}, {"role": "user", "content": TASK}]
    path, done = [], []
    for _ in range(5):
        m = client.chat(model=MODEL, messages=msgs, tools=SCHEMAS + [FINAL_TOOL], options=opts).message
        calls = m.tool_calls or []
        if not calls or any(c.function.name == "final_answer" for c in calls):
            break
        msgs.append(m)
        for c in calls:
            obs = call_tool(c.function.name, c.function.arguments)
            path.append(f"{c.function.name}({','.join(f'{k}={v}' for k, v in sorted(c.function.arguments.items()))})")
            msgs.append({"role": "tool", "tool_name": c.function.name, "content": json.dumps(obs)})
            done.append(f"{c.function.name} -> {json.dumps(obs)[:120]}")
        msgs.append({"role": "user", "content": nudge(TASK, done)})
    return " → ".join(path) + f"  | booked={[b['ref'] for b in STATE['bookings']]}"

for temp in (0.0, 0.8):
    runs = [trajectory(temp, seed) for seed in (1, 2, 3)]
    print(f"temperature={temp}: {len(set(runs))} distinct trajectory(ies) out of 3")
    for r in runs:
        print("   ", r[:200])

temperature=0.0: 1 distinct trajectory(ies) out of 3
    search_flights(date=2026-11-05,destination=CXB,origin=DAC) → book_flight(flight_id=BG101,passenger_name=Rahim)  | booked=['BK-BG101-RAHIM']
    search_flights(date=2026-11-05,destination=CXB,origin=DAC) → book_flight(flight_id=BG101,passenger_name=Rahim)  | booked=['BK-BG101-RAHIM']
    search_flights(date=2026-11-05,destination=CXB,origin=DAC) → book_flight(flight_id=BG101,passenger_name=Rahim)  | booked=['BK-BG101-RAHIM']


temperature=0.8: 1 distinct trajectory(ies) out of 3
    search_flights(date=2026-11-05,destination=CXB,origin=DAC) → book_flight(flight_id=BG101,passenger_name=Rahim)  | booked=['BK-BG101-RAHIM']
    search_flights(date=2026-11-05,destination=CXB,origin=DAC) → book_flight(flight_id=BG101,passenger_name=Rahim)  | booked=['BK-BG101-RAHIM']
    search_flights(date=2026-11-05,destination=CXB,origin=DAC) → book_flight(flight_id=BG101,passenger_name=Rahim)  | booked=['BK-BG101-RAHIM']


আমাদের run-এ temperature 0 **এবং 0.8** — দুটোতেই ৩টা run-এর trajectory হুবহু এক। এই ছোট, স্পষ্ট task-এ model-এর পছন্দ এত নিশ্চিত যে sampling-ও বদলায় না। শিক্ষা: দুই-তিনটা একই রকম run দেখে "agent deterministic" ধরে নেবেন না — কঠিন বা অস্পষ্ট task-এ, লম্বা trajectory-তে পার্থক্য জমে। কিন্তু মনে রাখুন — Thinking Machines-এর পরীক্ষায় server-এ temperature 0-তেও ১০০০ run-এ ৮০টা আলাদা output এসেছিল (batch-size-নির্ভর floating point)। তাই: **tool deterministic, test-এ LLM mock, live-এ pass^k।**

## 9. Mini Evaluation — final answer না, end state

τ-bench-এর মতো: run শেষে booking state expected-এর সাথে মেলে কিনা। সাথে policy: শুধু-খোঁজার প্রশ্নে `book_flight` ডাকাই ভুল।

In [13]:
GOLDEN = [
    (TASK, {("BG101", "Rahim")}, False),
    ("Which flights go from DAC to CGP on 2026-11-05?", set(), True),
    ("What is the cheapest flight from CXB to DAC?", set(), True),
    ("Book flight BG102 for Karim", set(), False),
]

def eval_agent(agent_fn):
    rows = []
    for prompt, expected, search_only in GOLDEN:
        answer, n = agent_fn(prompt)
        got = {(b["flight_id"], b["passenger"]) for b in STATE["bookings"]}
        rows.append({"task": prompt[:45], "end_state_ok": got == expected, "llm_calls": n, "answer": answer[:70]})
    return rows

import pandas as pd
print("text ReAct:")
display(pd.DataFrame(eval_agent(lambda p: text_react(p, verbose=False))))
print("native + final_answer + nudge:")
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    native_rows = eval_agent(fixed_native)
display(pd.DataFrame(native_rows))

text ReAct:


,task,end_state_ok,llm_calls,answer
0,Book the cheapest DAC to CXB flight on 2026-1,True,3,The booking reference for Rahim's flight from ...
1,Which flights go from DAC to CGP on 2026-11-0,False,6,Stopped: max_steps
2,What is the cheapest flight from CXB to DAC?,True,6,Stopped: max_steps
3,Book flight BG102 for Karim,True,6,Stopped: max_steps


native + final_answer + nudge:


,task,end_state_ok,llm_calls,answer
0,Book the cheapest DAC to CXB flight on 2026-1,True,4,Flight booked successfully for Rahim on BG101.
1,Which flights go from DAC to CGP on 2026-11-0,True,6,Stopped: max_steps
2,What is the cheapest flight from CXB to DAC?,True,3,"The cheapest flight from CXB to DAC is BG501, ..."
3,Book flight BG102 for Karim,True,6,Stopped: max_steps


**লক্ষ্য করুন:** কিছু row-তে `end_state_ok = True` কিন্তু answer `Stopped: max_steps` — booking state ঠিক আছে (কিছু বুক হয়নি), অথচ agent user-কে কোনো উত্তরই দেয়নি। শুধু end state দেখলে এগুলো "pass"। তাই project-এর harness end state-এর সাথে stop reason, নিষিদ্ধ tool call আর answer-ও grade করে — আর lecture-এর এই ছোট loop-এ repeat guard না থাকায় সেটা max_steps পর্যন্ত ঘুরেছে।

## 10. ReAct বনাম Plan-First (ReWOO-style)

ReWOO পুরো plan **একবারে** বানায় (`#E1`, `#E2` variable সহ), তারপর worker-রা চালায় — LLM call কম (paper: ৫× কম token)। দাম: planner মাঝের অপ্রত্যাশিত observation দেখে না। `format=` দিয়ে একটা plan নিই, তারপর deterministic ভাবে চালাই।

In [14]:
class PlanStep(BaseModel):
    tool: Literal["search_flights", "book_flight"]
    args: dict = Field(description='For book_flight you may use "#E1.cheapest_available" as flight_id')

class Plan(BaseModel):
    steps: list[PlanStep]

plan_msg = client.chat(model=MODEL, options=OPTIONS, format=Plan.model_json_schema(), messages=[
    {"role": "system", "content": "Write a complete plan of tool calls for the request. Tools: search_flights(origin, destination, date), "
                                  "book_flight(flight_id, passenger_name). Refer to the first search's cheapest flight with seats as "
                                  "\"#E1.cheapest_available\"."},
    {"role": "user", "content": TASK}]).message.content
plan = Plan.model_validate_json(plan_msg)
print("PLAN (1 LLM call):")
for i, s in enumerate(plan.steps, 1):
    print(f"  #E{i} = {s.tool}({s.args})")

STATE["bookings"].clear()
evidence = {}
for i, s in enumerate(plan.steps, 1):
    args = dict(s.args)
    if args.get("flight_id") == "#E1.cheapest_available":
        args["flight_id"] = next((f["id"] for f in evidence[1].get("flights", []) if f["seats_left"] > 0), "none")
    evidence[i] = call_tool(s.tool, args)
    print(f"  ran #E{i}: {json.dumps(evidence[i])[:110]}")
print("bookings:", STATE["bookings"])

PLAN (1 LLM call):
  #E1 = search_flights({'origin': 'DAC', 'destination': 'CXB', 'date': '2026-11-05'})
  #E2 = book_flight({'flight_id': '#E1.cheapest_available', 'passenger_name': 'Rahim'})
  ran #E1: {"ok": true, "flights": [{"id": "BG101", "departs": "08:00", "price_bdt": 6500, "seats_left": 5}, {"id": "US10
  ran #E2: {"ok": true, "booking_ref": "BK-BG101-RAHIM", "flight_id": "BG101", "passenger": "Rahim"}
bookings: [{'ref': 'BK-BG101-RAHIM', 'flight_id': 'BG101', 'passenger': 'Rahim'}]


| Pattern | LLM call (এই task) | সুবিধা | দুর্বলতা |
| --- | --- | --- | --- |
| ReAct (native + fix) | ~৩ | প্রতিটা observation দেখে মানিয়ে নেয় | প্রতি step-এ LLM call, loop-এর ঝুঁকি |
| ReWOO-style plan-first | ১ (+ deterministic execution) | কম call, দ্রুত, অনুমেয় | Plan ভুল বা পরিস্থিতি বদলালে (sold out) মানাতে পারে না |

## 🧠 Brain Teasers

1. Section 1-এর cell-এ "Action Input"-এর পরে একটা নকল `Observation: {"ok": true, "flights": []}` assistant message হিসেবে যোগ করে আবার চালান (stop ছাড়া)। Model কি সেই নকল data বিশ্বাস করে "কোনো ফ্লাইট নেই" বলে? বাস্তব agent-এ এর বিপদ কী?
2. Section 5-এর `guarded_loop`-এ `json.dumps(args, sort_keys=True)`-এর `sort_keys` সরিয়ে দিন। কোন case-এর status বদলায়?
3. Section 10-এর plan-first agent-কে "Book flight BG102 for Karim" দিন (sold out)। ReAct-এর তুলনায় কী হয়?
4. Section 8 আবার চালান `num_ctx=2048` দিয়ে। Trajectory বদলায়?